執行sql/01_data_cleaning.sql前需先在 BigQuery 建立目標 dataset

In [ ]:
CREATE SCHEMA IF NOT EXISTS traffic_ad_roi_clean;

| 步驟                           | 目的                           |
| ---------------------------- | ---------------------------- |
| Step 1 — Validation Checks   | 執行前先檢查，輸出 QA 報表              |
| Step 2 — Clean & Standardise | 清洗後寫入 traffic_ad_roi_clean.* |
| Step 3 — Summary Report      | 對比 raw vs clean 行數，確認結果      |

清洗邏輯說明
每張表均涵蓋以下清洗處理：

去重：ROW_NUMBER() OVER (PARTITION BY <pk>) 保留最新一筆，處理重複 primary key

NULL / 空值處理：COALESCE + NULLIF(TRIM(...), '') 統一填補預設值

Channel 標準化：CASE UPPER(TRIM(channel)) 將 GOOGLE、GOOGLE ADS 等變體統一為受控詞彙，對應你的 campaigns 表原有值

負數數值修正：GREATEST(..., 0) 防止 impressions、clicks、spend_usd 出現負數

CTR 重算：從原始 clicks/impressions 重新計算，比直接信任儲存值更可靠

衍生欄位：新增 is_active（campaigns）、cost_per_click_usd（ad_impressions）、engagement_tier（sessions）、order_value_tier（conversions）方便下游分析

***


## 資料規模總覽

| 表名 | 行數 | 類型 |
|---|---|---|
| `sessions` | **511,797** | 事實表（最大） |
| `conversions` | 18,288 | 事實表 |
| `ad_impressions` | 3,720 | 事實表 |
| `campaigns` | **12** | 維度表（只有12個活動） |
| `v_campaign_daily_ctr_cvr` | 2,988 | 每日時序 view |
| `v_monthly_channel_trend` | 60 | 月度 view |
| `v_monthly_roi_trend` | 36 | 月度 ROI view |
| `v_campaign_roi` | 12 | 活動彙總 view |
| `v_campaign_ctr_cvr_scatter` | 10 | 散點圖 view |
| `v_campaign_type_roi` | 11 | 類型彙總 view |
| `v_channel_performance` | 5 | 渠道彙總 view |
| `v_device_channel_conversion` | 15 | 裝置×渠道 view |
| `v_ctr_bucket_analysis` | 6 | CTR 分桶 view |



***

## 12 個 Campaign 分析

資料涵蓋 2024 年全年，共 5 個渠道、12 個活動 ：

| 渠道 | 活動數 | 預算範圍/日 |
|---|---|---|
| Google Ads | 4 | $300–$800 |
| Facebook Ads | 3 | $350–$700 |
| Email | 3 | $30–$80 |
| Organic | 1 | $0 |
| Direct | 1 | $0 |

***

## 關鍵業務洞察（來自 v_campaign_roi & v_channel_performance）

### ROAS 排名（由高至低）

| 活動 | 渠道 | ROAS | ROI % |
|---|---|---|---|
| Email_Abandoned_Cart | Email | **45.12** | 4,412% |
| Email_Newsletter_Monthly | Email | 27.86 | 2,686% |
| Email_Promo_Flash_Sale | Email | 26.95 | 2,595% |
| Google_Shopping_Q1 | Google Ads | 3.54 | 254% |
| Facebook_Retargeting | Facebook Ads | 2.63 | 163% |
| Google_Nonbrand_Search | Google Ads | 2.17 | 117% |
| Google_Brand_Search | Google Ads | 2.12 | 112% |
| Facebook_Awareness | Facebook Ads | 1.08 | 8% |
| Facebook_Conversion | Facebook Ads | 1.04 | 4% |
| **Google_Display_Remarketing** | Google Ads | **0.70** | **-30%** ⚠️ |

> ⚠️ `Google_Display_Remarketing` 是唯一**虧損活動**（ROI -30%），值得在 Power BI 重點標示。

### 渠道效率對比 

| 渠道 | ROAS | CVR | CPA |
|---|---|---|---|
| Email | **30.8** | 7.4% | $3.36 |
| Google Ads | 2.15 | 3.5% | $44.45 |
| Facebook Ads | 1.38 | 2.7% | $64.08 |
| Direct | N/A | 6.8% | $0 |
| Organic | N/A | 1.9% | $0 |

